In [ ]:
!pip install langextract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 4.7 MB/s eta 0:00:00


In [ ]:
dataset_folder="path/2/dataset"
texts_folder="path/2/raw_texts"

In [ ]:
project_folder="path/2/project"
LANGUAGES=["PT","BG","EN","RU"]

##Get Data

In [ ]:
import json
import os
with open(os.path.join(dataset_folder,"train.json"), "r", newline="") as f:
    train=json.load(f)


with open(os.path.join(dataset_folder,"val.json"), "r", newline="") as f:
    val=json.load(f)
with open(os.path.join(dataset_folder,"test.json"), "r", newline="") as f:
    test=json.load(f)

In [ ]:
train[0:4]

#PreComputation of Embeddings

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer('sentence-transformers/LaBSE')


In [ ]:
def computeEmbeddings(text):
  embeddings = model.encode(text)
  return embeddings




In [ ]:
def computeEmbeddingsAllDocuments(train):
  data_emb=list()
  for entry in train:
    #print(id)
    #print(row)
    entry_emb=entry
    embeddings=computeEmbeddings(entry["text"])
    entry_emb["embeddings"]=embeddings
    data_emb.append(entry_emb)

  return data_emb




In [ ]:
import pickle
#dump embeddings_df in a json file
embeddings_df_test=computeEmbeddingsAllDocuments(test)
embeddings_df_val=computeEmbeddingsAllDocuments(val)
embeddings_df_train=computeEmbeddingsAllDocuments(train)
pickle.dump(embeddings_df_test,open(os.path.join(dataset_folder,"embeddings_test.p"), "wb"))
pickle.dump(embeddings_df_train,open(os.path.join(dataset_folder,"embeddings_train.p"), "wb"))
pickle.dump(embeddings_df_val,open(os.path.join(dataset_folder,"embeddings_val.p"), "wb"))



In [ ]:
embeddings_df_train[0:10]

#Compute SimilarityMaps

In [ ]:
import pickle


train_embeddings=pickle.load(open(os.path.join(dataset_folder,"embeddings_train.p"), "rb"))
val_embeddings=pickle.load(open(os.path.join(dataset_folder,"embeddings_val.p"), "rb"))
test_embeddings=pickle.load(open(os.path.join(dataset_folder,"embeddings_test.p"), "rb"))

In [ ]:
from sentence_transformers import util

def findMostSimilarEmbedding(test_entry,train_embeddings):
  highest_similarity=0
  most_similar_id=-1
  for train_entry in train_embeddings:
    similarity=util.cos_sim(test_entry["embeddings"],train_entry["embeddings"]).item()
    #print(similarity)
    if similarity>highest_similarity:
      #print(similarity,highest_similarity)
      highest_similarity=similarity
      most_similar_id=train_entry["doc_id"]
      most_similar_lang=train_entry["language"]
      most_similar_dom_narr=train_entry["dom_narr"]




  return highest_similarity,most_similar_id,most_similar_lang,most_similar_dom_narr

The precomputeSimilarites creates the mapping between test documents and the most similar train document. Does not have into account the narratives

In [ ]:
def precomputeSimilarities(test):
  similarity_map=list()
  for entry in test:
    temp=dict()
    temp["language"]=entry["language"]
    temp["doc_id"]=entry["doc_id"]
    temp["language"]=entry["language"]
    temp["dom_narr"]=entry["dom_narr"]
    similarity,most_similar_id,most_similar_lang,most_similar_dom_narr=findMostSimilarEmbedding(entry,train_embeddings)
    temp["train_most_similar_id"]=most_similar_id

    temp["train_most_similar_id_lang"]=most_similar_lang
    temp["train_most_similar_id_dom_narr"]=most_similar_dom_narr

    temp["similarity"]=similarity
    similarity_map.append(temp)

  return similarity_map





In [ ]:
similarity_map=precomputeSimilarities(test_embeddings)

In [ ]:
import pandas as pd
similarity_map_df=pd.DataFrame.from_dict(similarity_map)

In [ ]:
similarity_map[0]

{'language': 'PT',
 'doc_id': 'PT_318.txt',
 'dom_narr': 'CC: Amplifying Climate Fears',
 'train_most_similar_id': 'PT_389.txt',
 'train_most_similar_id_lang': 'PT',
 'train_most_similar_id_dom_narr': 'CC: Amplifying Climate Fears',
 'similarity': 0.7272565960884094}

In [ ]:
similarity_map_df.to_csv(os.path.join(dataset_folder,"similarity_map.csv"))

In [ ]:

def findMostSimilarEmbeddingwNarrative(test_entry,train_embeddings):
  highest_similarity=0
  most_similar_id=-1

  for train_entry in train_embeddings:
    if train_entry["dom_narr"]==test_entry["dom_narr"]:
      h_embeddings=train_entry["embeddings"]
      similarity=util.cos_sim(test_entry["embeddings"],h_embeddings).item()
      #print(similarity)
      if similarity>highest_similarity:
        #print(similarity,highest_similarity)
        highest_similarity=similarity
        most_similar_id=train_entry["doc_id"]
        most_similar_lang=train_entry["language"]
        most_similar_dom_narr=train_entry["dom_narr"]

  if most_similar_id==-1:
    #print("no narrative found. falling back on similarity")
    return  findMostSimilarEmbedding(test_entry,train_embeddings)
  #print("found NARRATIVE")
  return highest_similarity,most_similar_id,most_similar_lang,most_similar_dom_narr

In [ ]:
def precomputeSimilaritieswNarrative(test):
  similarity_map=list()
  for entry in test:
    temp=dict()
    temp["language"]=entry["language"]
    temp["doc_id"]=entry["doc_id"]
    temp["language"]=entry["language"]
    temp["dom_narr"]=entry["dom_narr"]
    similarity,most_similar_id,most_similar_lang,most_similar_dom_narr=findMostSimilarEmbeddingwNarrative(entry,train_embeddings)
    temp["train_most_similar_id"]=most_similar_id

    temp["train_most_similar_id_lang"]=most_similar_lang
    temp["train_most_similar_id_dom_narr"]=most_similar_dom_narr

    temp["similarity"]=similarity
    similarity_map.append(temp)

  return similarity_map


In [ ]:
similarity_map_narrative=precomputeSimilaritieswNarrative(test_embeddings)

In [ ]:
similarity_map_narrative_df=pd.DataFrame.from_dict(similarity_map_narrative)




In [ ]:
similarity_map_df

,language,doc_id,dom_narr,train_most_similar_id,train_most_similar_id_lang,train_most_similar_id_dom_narr,similarity
0,PT,PT_318.txt,CC: Amplifying Climate Fears,PT_389.txt,PT,CC: Amplifying Climate Fears,0.727257
1,BG,A6_CC_BG_10380.txt,CC: Amplifying Climate Fears,A6_CC_BG_10556.txt,BG,CC: Amplifying Climate Fears,0.777933
2,PT,PT_CC_427.txt,CC: Amplifying Climate Fears,PT_401.txt,PT,CC: Criticism of climate policies,0.774226
3,BG,A6_CC_BG_8984.txt,CC: Amplifying Climate Fears,BG_364.txt,BG,CC: Amplifying Climate Fears,0.813909
4,PT,PT_346.txt,CC: Amplifying Climate Fears,A6_CC_BG_13641.txt,BG,CC: Amplifying Climate Fears,0.845797
...,...,...,...,...,...,...,...
113,RU,RU-URW-1322.txt,URW: Russia is the Victim,RU-URW-1134.txt,RU,"URW: Discrediting the West, Diplomacy",0.694951
114,PT,PT_95.txt,URW: Russia is the Victim,PT_URW_TEST_494.txt,PT,"URW: Discrediting the West, Diplomacy",0.749229
115,BG,A4_URW_BG_15138.txt,URW: Speculating war outcomes,A4_URW_BG_15605.txt,BG,URW: Speculating war outcomes,0.681898
116,EN,EN_UA_DEV_100002.txt,URW: Speculating war outcomes,EN_UA_101805.txt,EN,URW: Hidden plots by secret schemes of powerfu...,0.762570


In [ ]:
similarity_map_narrative_df.to_csv(os.path.join(dataset_folder,"similarity_map_narrative.csv"))

In [ ]:
#similarity_map_df.to_csv(os.path.join(dataset_folder,"similarity_map.csv"))

#Dynamic Few Shot

In [ ]:
dataset_folder="path/2/dataset"



In [ ]:
import pandas as pd
import os
import pickle
similarity_map_df=pd.read_csv(os.path.join(dataset_folder,"similarity_map.csv"))


In [ ]:
train_embeddings=pickle.load(open(os.path.join(dataset_folder,"embeddings_train.p"), "rb"))
test_embeddings=pickle.load(open(os.path.join(dataset_folder,"embeddings_test.p"), "rb"))
val_embeddings=pickle.load(open(os.path.join(dataset_folder,"embeddings_val.p"), "rb"))

In [ ]:
similarity_map_df

,Unnamed: 0,language,doc_id,dom_narr,train_most_similar_id,train_most_similar_id_lang,train_most_similar_id_dom_narr,similarity
0,0,PT,PT_318.txt,CC: Amplifying Climate Fears,PT_389.txt,PT,CC: Amplifying Climate Fears,0.727257
1,1,BG,A6_CC_BG_10380.txt,CC: Amplifying Climate Fears,A6_CC_BG_10556.txt,BG,CC: Amplifying Climate Fears,0.777933
2,2,PT,PT_CC_427.txt,CC: Amplifying Climate Fears,PT_401.txt,PT,CC: Criticism of climate policies,0.774226
3,3,BG,A6_CC_BG_8984.txt,CC: Amplifying Climate Fears,BG_364.txt,BG,CC: Amplifying Climate Fears,0.813909
4,4,PT,PT_346.txt,CC: Amplifying Climate Fears,A6_CC_BG_13641.txt,BG,CC: Amplifying Climate Fears,0.845797
...,...,...,...,...,...,...,...,...
113,113,RU,RU-URW-1322.txt,URW: Russia is the Victim,RU-URW-1134.txt,RU,"URW: Discrediting the West, Diplomacy",0.694951
114,114,PT,PT_95.txt,URW: Russia is the Victim,PT_URW_TEST_494.txt,PT,"URW: Discrediting the West, Diplomacy",0.749229
115,115,BG,A4_URW_BG_15138.txt,URW: Speculating war outcomes,A4_URW_BG_15605.txt,BG,URW: Speculating war outcomes,0.681898
116,116,EN,EN_UA_DEV_100002.txt,URW: Speculating war outcomes,EN_UA_101805.txt,EN,URW: Hidden plots by secret schemes of powerfu...,0.762570


In [ ]:
import langextract as lx

def getEvidenceFromPosition(text,list_positions):
  evidences=list()
  for entry in list_positions:
    begin=entry["begin"]
    end=entry["end"]
    evidence=text[begin:end]
    evidences.append(evidence)
  return evidences


def getExamples(doc_id,train_dataset):
  #print(doc_id)
  extractions=[x["evidence"] for x in train_dataset if x["doc_id"]==doc_id][0]
  text=[x["text"] for x in train_dataset if x["doc_id"]==doc_id][0]
  #print(extractions)
  extractions=getEvidenceFromPosition(text,extractions)

  #print(extractions)
  lx_extract=list()
  for ex in extractions:
    lx_extract.append(lx.data.Extraction(extraction_class="evidence", extraction_text=ex),)

  narrative=[x["dom_narr"] for x in train_dataset if x["doc_id"]==doc_id][0]
  lang=[x["language"] for x in train_dataset if x["doc_id"]==doc_id][0]

  #print(narrative,lang)
  # Define example data with entities in order of appearance
  examples = [
      lx.data.ExampleData(
          text="NARRATIVE: " + narrative + "\n\n TEXT:" + text,
          extractions=lx_extract
      )
  ]
  return examples

In [ ]:
id_ex=train_embeddings[0]["doc_id"]
res_temp=getExamples(id_ex,train_embeddings)
res_temp

The getDynamicExample function gets the most similar example without having into account the narrative. Works well in teh cases of open narrative scenarios

In [ ]:
def getDynamicExample(doc_id,train_dataset,similarity_map_df):
  matching=similarity_map_df[similarity_map_df["doc_id"]==doc_id].iloc[0]
  most_similar_id=matching["train_most_similar_id"]
  examples=getExamples(most_similar_id,train_dataset)
  return examples


In [ ]:
getDynamicExample(test_embeddings[0]["doc_id"],train_embeddings,similarity_map_df)

In [ ]:
import langextract as lx
LANGEXTRACT_API_KEY="key2gemini"
#gemma-4-31b-it
#gemma-4-26b-a4b-it
def extract_info_gemini_models(example_annotations,input_text,input_narrative,model="gemini-2.5-flash-lite"):

  # Define extraction prompt
  prompt_description = "Extract all text spans that support and justify the selected dominant narrative: \n \n  DOMINANT NARRATIVE:"+ input_narrative

  result = lx.extract(
    text_or_documents=input_text,
    prompt_description=prompt_description,
    examples=example_annotations,
    api_key=LANGEXTRACT_API_KEY,
    model_id=model
)
  return result


In [ ]:
input_text=test_embeddings[0]["text"]
input_narrative=test_embeddings[0]["dom_narr"]
annotations_examples=getDynamicExample(test_embeddings[0]["doc_id"],train_embeddings,similarity_map_df)
res=extract_info_iaedu(annotations_examples,input_text,input_narrative)


LangExtract: Processing, current=2,642 chars, processed=0 chars:  [00:10]


In [ ]:
res

In [ ]:
lx.io.save_annotated_documents([res], output_name=str(test_embeddings[0]["doc_id"])+".json", output_dir=".")
html_content = lx.visualize(os.path.join(".",str(test_embeddings[0]["doc_id"])+".json"))
with open(os.path.join(".",str(test_embeddings[0]["doc_id"])+".html"), "w") as f:
    if hasattr(html_content, 'data'):
        f.write(html_content.data)  # For Jupyter/Colab
    else:
        f.write(html_content)

print("processed!!!",test_embeddings[0]["doc_id"])

LangExtract: Saving to PT_318.txt.json: 1 docs [00:00, 590.75 docs/s]

✓ Saved 1 documents to PT_318.txt.json



LangExtract: Loading PT_318.txt.json: 100%|██████████| 8.87k/8.87k [00:00<00:00, 4.74MB/s]

✓ Loaded 1 documents from PT_318.txt.json
processed!!! PT_318.txt


#ALL Data

In [ ]:
test_embeddings

,Unnamed: 0,language,doc_id,dom_narr,train_most_similar_id,train_most_similar_id_lang,train_most_similar_id_dom_narr,similarity
0,0,PT,PT_318.txt,CC: Amplifying Climate Fears,PT_389.txt,PT,CC: Amplifying Climate Fears,0.727257
1,1,BG,A6_CC_BG_10380.txt,CC: Amplifying Climate Fears,A6_CC_BG_10556.txt,BG,CC: Amplifying Climate Fears,0.777933
2,2,PT,PT_CC_427.txt,CC: Amplifying Climate Fears,PT_401.txt,PT,CC: Criticism of climate policies,0.774226
3,3,BG,A6_CC_BG_8984.txt,CC: Amplifying Climate Fears,BG_364.txt,BG,CC: Amplifying Climate Fears,0.813909
4,4,PT,PT_346.txt,CC: Amplifying Climate Fears,A6_CC_BG_13641.txt,BG,CC: Amplifying Climate Fears,0.845797
...,...,...,...,...,...,...,...,...
113,113,RU,RU-URW-1322.txt,URW: Russia is the Victim,RU-URW-1134.txt,RU,"URW: Discrediting the West, Diplomacy",0.694951
114,114,PT,PT_95.txt,URW: Russia is the Victim,PT_URW_TEST_494.txt,PT,"URW: Discrediting the West, Diplomacy",0.749229
115,115,BG,A4_URW_BG_15138.txt,URW: Speculating war outcomes,A4_URW_BG_15605.txt,BG,URW: Speculating war outcomes,0.681898
116,116,EN,EN_UA_DEV_100002.txt,URW: Speculating war outcomes,EN_UA_101805.txt,EN,URW: Hidden plots by secret schemes of powerfu...,0.762570


#Results

In [ ]:
#gemma-4-31b-it
#gemma-4-26b-a4b-it
#gpt4o
import os
from tqdm import tqdm
import time
output_folder="path/2/results"
def process_test_set(train,test,experiment_type,model,similarity_map):
  experiment_folder=os.path.join(output_folder,experiment_type,model)
  if(not experiment_folder):
    os.makedirs(os.path.join(output_folder,model))
    print("folder created")
  for entry in tqdm(test):

    file_check=os.path.join(experiment_folder,str(entry["doc_id"])+".json")
    print(file_check)
    if(os.path.exists(file_check)):
      print("file already exists")
      continue


    examples= getDynamicExample(entry["doc_id"],similarity_map_df=similarity_map,train_dataset=train)
    input_text=entry["text"]
    input_narrative=entry["dom_narr"].split(":")[1]
    if(model=="gpt4o"):
      #time.sleep(30)
      res=extract_info_iaedu(examples,input_text,input_narrative)
    else:
      res=extract_info_gemini_models(examples,input_text,input_narrative,model)

    print("processed!!!",entry["doc_id"])

    lx.io.save_annotated_documents([res], output_name=str(entry["doc_id"])+".json", output_dir=experiment_folder)
    html_content = lx.visualize(os.path.join(experiment_folder,str(entry["doc_id"])+".json"))

    with open(os.path.join(experiment_folder,str(entry["doc_id"])+".html"), "w") as f:
        if hasattr(html_content, 'data'):
            f.write(html_content.data)  # For Jupyter/Colab
        else:
            f.write(html_content)

    print("written!!!",entry["doc_id"])
    time.sleep(60)



In [ ]:
process_test_set(train=train_embeddings,
                 test=test_embeddings,
                 experiment_type="narrative_train",
                 model="gemini-2.5-flash",
                 similarity_map=similarity_map_df)

In [ ]:
len(set(test["id"]))


226

#Debugging

In [ ]:
#For Debug Can be useful for further testing
for ind,row in test.iterrows():
  try:
    examples=getDynamicExample(row["id"],lang=row["language"],similarity_map_df=similarity_map_df,train_dataset=train)
  except Exception as e:
    print(row["id"],"FAILED",str(e))
    continue
  else:
    print(row["id"],"Success")


In [ ]:
any(similarity_map_df["id"]=="RU-URW-1139.txt")

True